In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname('__file__'), '..') if '__file__' in dir() else os.path.abspath('..'))

import torch
from rl.curricula import Curriculum
from rl.policy import Policy
from rl.ppo import train_epoch
from tqdm import tqdm

In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

window_r = 10
policy = Policy(window_radius=window_r).to(device)
optimizer = torch.optim.Adam(policy.parameters(), lr=2.5e-4, eps=1e-5)

In [ ]:
stages = [
    Curriculum(num_envs=2048, window_radius=window_r, num_agents=1, max_steps=50, terr_delta_coef=0.5, death_penalty=-10.0, env_size=15, terr_delta_max=20),
    Curriculum(num_envs=256, window_radius=window_r, death_penalty=-20.0, terr_delta_min=-20, kill_reward=50, leave_terr_bonus=0.1),
    Curriculum(num_envs=256, window_radius=window_r, terr_delta_min=-100, terr_delta_max=100)
]

stage = 0
curriculum = stages[stage]

best_reward = -float('inf')
stall_count = 0
patience = 25

os.makedirs("checkpoints", exist_ok=True)

pbar = tqdm(range(5000))
for iteration in pbar:
    metrics = train_epoch(curriculum, policy, optimizer, device=device)
    reward = metrics['mean_reward']

    if reward > best_reward:
        best_reward = reward
        stall_count = 0
        torch.save(policy.state_dict(), f"checkpoints/stage_{stage + 1}_best.pt")
    else:
        stall_count += 1

    stage_str = f"stg {stage + 1}/{len(stages)}"
    pbar.set_postfix_str(
        f"loss={metrics['policy_loss']:.3f} "
        f"v_loss={metrics['value_loss']:.3f} "
        f"ent={metrics['entropy']:.3f} "
        f"rew={reward:.3f} "
        f"ev={metrics['explained_variance']:.2f} "
        f"stall={stall_count}/{patience} {stage_str}"
    )

    if stall_count >= patience:
        stage += 1
        if stage >= len(stages):
            break
        curriculum = stages[stage]
        best_reward = -float('inf')
        stall_count = 0